# Figure 4 Pareto and IoH Panels

This notebook regenerates the Figure 4 Bayesian-optimization Pareto and IoH panels from the authoritative JSON-derived 112-sample BO table. The x-axis uses RHE-converted potential at -10 mA/cm² in mV, following the original figure convention.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator, FuncFormatter
from shapely.geometry import Polygon, box
from shapely.ops import unary_union

base = Path.cwd()
data_path = base / "curated" / "figure4_bo_complete_112.csv"
out_path = base / "eight_panels_2x4_batch1_7_lg.png"

df_all = pd.read_csv(data_path)

systems = ["Co-Mo", "Co-V", "Co-Ni", "Fe-Ni"]
voltage_key = "potential10_mV_vs_RHE"
source_voltage_key = "lsv_overpotential_-0.01"
sep_key = "separation_coefficient"
RHE_SHIFT_V = 0.0592 * 14 + 0.098

TOP_FIGSIZE_PER_PANEL = (8, 9)
S_SCATTER = 100
SCATTER_ALPHA = 0.95
AREA_ALPHA = 0.05
HV_LINE_ALPHA_EARLY = 0.80
HV_LINE_ALPHA_LAST = 0.80
HV_LINE_LW = 2.0
PARETO_LINE_ALPHA_LAST = 0.80
PARETO_LINE_LW = 2.5
PARETO_POINT_SIZE = 180
PARETO_POINT_ALPHA = 0.35
REVERSE_RAINBOW = True
FRAME_LW = 2.5
LINE_LW = 2.5
LINE_ALPHA = 0.85
SCATTER_S = 300
HIGHLIGHT_S = 300


def fix_geom(g):
    if g is None:
        return Polygon()
    if not g.is_valid:
        g = g.buffer(0)
    return g


def iter_polygons(g):
    if g is None or g.is_empty:
        return
    if g.geom_type == "Polygon":
        yield g
    elif g.geom_type == "MultiPolygon":
        yield from g.geoms
    elif g.geom_type == "GeometryCollection":
        for gg in g.geoms:
            yield from iter_polygons(gg)


def draw_geom_boundary(ax, geom, color, alpha=0.3, lw=2.0):
    for p in iter_polygons(geom):
        x, y = p.exterior.xy
        ax.plot(x, y, color=color, alpha=alpha, linewidth=lw)


def fill_geom(ax, geom, color, alpha=0.05):
    for p in iter_polygons(geom):
        x, y = p.exterior.xy
        ax.fill(x, y, color=color, alpha=alpha, linewidth=0)


def load_system_df(sys_name):
    df = df_all[df_all["system"] == sys_name].copy()
    df[voltage_key] = 1000 * (df[source_voltage_key] + RHE_SHIFT_V)
    df = df.dropna(subset=[voltage_key, sep_key]).copy()
    df["row_id"] = df["row_id"].astype(int)
    df = df.sort_values("row_id").reset_index(drop=True)
    df["batch"] = (df["row_id"] // 4) + 1
    max_batch = int(df["batch"].max())
    x_ref = float(df[voltage_key].min())
    y_ref = float(df[sep_key].min())
    return df, max_batch, x_ref, y_ref


def pareto_front_max_x_max_y(df_in, x=voltage_key, y=sep_key):
    s = df_in.sort_values([x, y], ascending=[False, False]).reset_index(drop=True)
    best_y = -np.inf
    keep = []
    for _, r in s.iterrows():
        if r[y] > best_y + 1e-12:
            keep.append(True)
            best_y = r[y]
        else:
            keep.append(False)
    front = s.loc[keep].copy()
    return front.sort_values(x, ascending=True).reset_index(drop=True)


def hv_union_geom(front, x_ref, y_ref, x=voltage_key, y=sep_key):
    rects = []
    for _, r in front.iterrows():
        xi = float(r[x])
        yi = float(r[y])
        if xi <= x_ref or yi <= y_ref:
            continue
        rects.append(box(x_ref, y_ref, xi, yi))
    if not rects:
        return Polygon()
    return fix_geom(unary_union(rects))


def shiny_ball(ax, x, y, s=140, color=None, edgecolor="black", lw=1.0,
               highlight_alpha=0.35, zorder=20):
    if color is None:
        color = plt.rcParams["axes.prop_cycle"].by_key()["color"][0]
    ax.scatter([x], [y], s=s, color=color, edgecolors=edgecolor, linewidths=lw, zorder=zorder)
    offset_x = -0.002 if isinstance(x, (float, np.floating)) else -0.03
    offset_y = 0.06
    ax.scatter([x + offset_x], [y + offset_y], s=s * 0.22, color="white",
               alpha=highlight_alpha, linewidths=0, zorder=zorder + 10)


def batch_colors(max_batch):
    cmap = plt.cm.rainbow
    colors = [cmap(i / max(1, (max_batch - 1))) for i in range(max_batch)]
    return colors[::-1] if REVERSE_RAINBOW else colors


def plot_system(ax, sys_name):
    df, max_batch, x_ref, y_ref = load_system_df(sys_name)
    fronts, hv_geoms = {}, {}
    for b in range(1, max_batch + 1):
        cum = df[df["batch"] <= b].copy()
        fronts[b] = pareto_front_max_x_max_y(cum)
        hv_geoms[b] = hv_union_geom(fronts[b], x_ref, y_ref)

    colors = batch_colors(max_batch)
    for b in range(1, max_batch + 1):
        c = colors[b - 1]
        dfb = df[df["batch"] == b]
        ax.scatter(dfb[voltage_key], dfb[sep_key], s=S_SCATTER, color=c,
                   alpha=SCATTER_ALPHA, edgecolors="black", linewidths=0.8, zorder=110)

    c1 = colors[0]
    cl = colors[max_batch - 1]
    hv1 = fix_geom(hv_geoms[1])
    hvl = fix_geom(hv_geoms[max_batch])

    fill_geom(ax, hv1, c1, alpha=AREA_ALPHA)
    if max_batch > 1:
        fill_geom(ax, fix_geom(hvl.difference(hv1)), cl, alpha=AREA_ALPHA)
    draw_geom_boundary(ax, hv1, c1, alpha=HV_LINE_ALPHA_EARLY, lw=HV_LINE_LW)
    draw_geom_boundary(ax, hvl, cl, alpha=HV_LINE_ALPHA_LAST, lw=HV_LINE_LW)

    for front, color, z in [(fronts[1], c1, 120), (fronts[max_batch], cl, 130)]:
        ax.scatter(front[voltage_key], front[sep_key], s=PARETO_POINT_SIZE, color=color,
                   edgecolors="black", linewidths=0.8, alpha=PARETO_POINT_ALPHA, zorder=z)
        ax.plot(front[voltage_key], front[sep_key], linewidth=PARETO_LINE_LW,
                color=color, alpha=PARETO_LINE_ALPHA_LAST, linestyle="--")

    ax.yaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_major_locator(MultipleLocator(50))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.0f}"))
    ax.tick_params(axis="both", which="both", labelsize=24)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontname("Arial")
        t.set_fontweight("bold")
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_linewidth(FRAME_LW)


def compute_hv_series(sys_name):
    df, max_batch, x_ref, y_ref = load_system_df(sys_name)
    batches = np.arange(1, max_batch + 1, dtype=int)
    hv = []
    for b in batches:
        cum = df[df["batch"] <= b].copy()
        front = pareto_front_max_x_max_y(cum)
        hv.append(float(fix_geom(hv_union_geom(front, x_ref, y_ref)).area))
    hv = np.asarray(hv, dtype=float)
    ioh = 100 * (hv - hv[0]) / hv[0] if abs(hv[0]) > 1e-12 else np.zeros_like(hv)
    return batches, ioh, hv


def plot_hv_panel(ax, sys_name):
    batches, y, _ = compute_hv_series(sys_name)
    main_color = "#FF42AD"
    ax.plot(batches, y, linewidth=LINE_LW, alpha=LINE_ALPHA, color=main_color, zorder=10)
    for b, yy in zip(batches, y):
        shiny_ball(ax, b, yy, s=SCATTER_S * 1.2, color=main_color, lw=1.0, zorder=20)
    shiny_ball(ax, batches[0], y[0], s=HIGHLIGHT_S, color=main_color, lw=0.0, zorder=30)
    shiny_ball(ax, batches[-1], y[-1], s=HIGHLIGHT_S, color=main_color, lw=0.0, zorder=31)
    ax.set_xticks(batches)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, pos: f"{v:.0f}"))
    ax.tick_params(axis="both", which="both", labelsize=24)
    for t in ax.get_xticklabels() + ax.get_yticklabels():
        t.set_fontname("Arial")
        t.set_fontweight("bold")
    ax.margins(x=0.05, y=0.10)
    for side in ["left", "right", "top", "bottom"]:
        ax.spines[side].set_linewidth(FRAME_LW)


ncol = len(systems)
fig, axs = plt.subplots(
    2, ncol,
    figsize=(TOP_FIGSIZE_PER_PANEL[0] * ncol, TOP_FIGSIZE_PER_PANEL[1] * 2),
    gridspec_kw={"height_ratios": [1.15, 1.00]},
)

for j, sys_name in enumerate(systems):
    ax_top = axs[0, j]
    ax_bot = axs[1, j]
    plot_system(ax_top, sys_name)
    plot_hv_panel(ax_bot, sys_name)
    if sys_name == "Co-Mo":
        ax_bot.set_ylim(top=18.2)
    ax_top.set_title(sys_name, fontname="Arial", fontweight="bold", fontsize=30, pad=14)
    ax_bot.set_title(sys_name, fontname="Arial", fontweight="bold", fontsize=30, pad=14)
    ax_top.set_xlabel("")
    ax_top.set_ylabel("")
    ax_bot.set_xlabel("")
    ax_bot.set_ylabel("")

fig.tight_layout(rect=[0.11, 0.06, 0.99, 0.94], h_pad=10.0)

top_axes = list(axs[0, :].ravel())
bot_axes = list(axs[1, :].ravel())
bbox_top = mpl.transforms.Bbox.union([ax.get_position() for ax in top_axes])
bbox_bot = mpl.transforms.Bbox.union([ax.get_position() for ax in bot_axes])
pad = 0.02

fig.text(
    bbox_top.x0 - 1.5 * pad,
    0.5 * (bbox_top.y0 + bbox_top.y1),
    "Separation Coefficient (@ -50 mA/cm²)",
    rotation="vertical",
    va="center",
    ha="center",
    fontname="Arial",
    fontweight="bold",
    fontsize=30,
)
fig.text(
    0.5 * (bbox_top.x0 + bbox_top.x1),
    0.5 * (bbox_top.y0 + bbox_bot.y1) + 1.2 * pad,
    "Potential (@ -10 mA/cm², vs. RHE, mV)",
    va="center",
    ha="center",
    fontname="Arial",
    fontweight="bold",
    fontsize=30,
)
fig.text(
    bbox_bot.x0 - 1.5 * pad,
    0.5 * (bbox_bot.y0 + bbox_bot.y1),
    "IoH (%)",
    rotation="vertical",
    va="center",
    ha="center",
    fontname="Arial",
    fontweight="bold",
    fontsize=30,
)
fig.text(
    0.5 * (bbox_bot.x0 + bbox_bot.x1),
    2.5 * pad,
    "Batch Number",
    va="center",
    ha="center",
    fontname="Arial",
    fontweight="bold",
    fontsize=30,
)

legend_colors = batch_colors(7)
legend_handles = [
    Line2D([0], [0], marker="o", linestyle="None", label=f"Batch {i}",
           markerfacecolor=legend_colors[i - 1], markeredgecolor="black",
           markeredgewidth=0.8, markersize=5.5)
    for i in range(1, 8)
]
legend_handles.append(
    Line2D([0], [0], color="purple", linestyle="--", linewidth=1.2, label="Pareto Front")
)
fig.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.985),
    ncol=8,
    frameon=True,
    fancybox=False,
    edgecolor="black",
    framealpha=1.0,
    prop={"family": "Arial", "weight": "bold", "size": 9},
    handlelength=1.3,
    handletextpad=0.45,
    columnspacing=0.85,
    borderpad=0.35,
    labelspacing=0.25,
)

fig.savefig(out_path, dpi=600, bbox_inches="tight", pad_inches=0.02)
plt.close(fig)

print(f"Saved: {out_path}")
for sys_name in systems:
    batches, ioh, hv = compute_hv_series(sys_name)
    print(f"{sys_name}: final IoH = {ioh[-1]:.2f}%")


Saved: C:\Users\yibinjiang\Documents\GitHub\HD_separation\0-Figures\Figure4\eight_panels_2x4_batch1_7_lg.png
Co-Mo: final IoH = 17.24%
Co-V: final IoH = 17.29%
Co-Ni: final IoH = 26.06%
Fe-Ni: final IoH = 56.09%
